# LexFrame quickstart

LexFrame reads a hidden state the way the model reads it: through the model's own remaining layers, with the context the state attends to, in the model's own vocabulary. This notebook reads SmolLM3-3B, compares LexFrame with the logit lens and the Jacobian lens, and steers a prediction. Choose a GPU runtime (Runtime > Change runtime type > T4 GPU).

[Paper](https://arxiv.org/abs/2610.22169) | [Blog post](https://jianguda.github.io/blogs/lexframe/) | [Code](https://github.com/jianguda/lexframe)

In [ ]:
!pip install -q lexframe datasets matplotlib

## Load a model and calibrate a frame

Any causal language model in Hugging Face Transformers works. The frame is calibrated once on unlabelled text, which takes a few minutes and is cached on disk.

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

import lexframe as lf

name = "HuggingFaceTB/SmolLM3-3B"
model = AutoModelForCausalLM.from_pretrained(name, dtype="auto", device_map="cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(name)
texts = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1", split="train")["text"][:1000]

frame = lf.LexFrame(model, tokenizer).init(texts)
frame

## Read, attribute and steer

`view` reads every layer at every position. The reading at the last position shows where the answer appears, `attr` shows which layers wrote it, and `steer` moves a state along the lexical basis of two tokens.

In [ ]:
text = "The Eiffel Tower is located in the city of"
v = frame.view(text, k=5)
for layer in v.layers:
    print(layer, v.tokens[layer, -1])

In [ ]:
print("Paris tops the reading from layer", v.first_layer(" Paris")[-1])
contribution = frame.attr(text)[:, -1]
print("the layers that wrote the prediction most:", contribution.argsort()[::-1][:3])

In [ ]:
c = frame.view(text, layers=16)
print("before:", frame.view(c).tokens[-1])
print("after: ", frame.view(c.steer({" Paris": -4, " London": 4})).tokens[-1])

## Compare with two baselines

The baselines live apart from the frame, in `lexframe.baselines`. The logit lens reads every layer with the output head; the Jacobian lens fits one linear map per layer. Both read every state of a layer alike, whatever its context. Fitting the Jacobian lens takes one backward pass per 8 hidden dimensions and sequence, which adds up on a free GPU: four sequences take several minutes here, and more sequences fit the lens better.

In [ ]:
from lexframe.baselines import JacobianLens, LogitLens

lenses = {"logit lens": LogitLens(model, tokenizer), "Jacobian lens": JacobianLens(model, tokenizer).fit(texts, size=4)}

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

words = [tokenizer.decode([i]) for i in tokenizer.encode(text, add_special_tokens=False)]
fig, axes = plt.subplots(1, 3, figsize=(18, 8), sharey=True, layout="constrained")
for ax, (title, method) in zip(axes, {**lenses, "LexFrame": frame}.items()):
    r = method.view(text)
    ax.imshow(np.nan_to_num(r.probs[..., 0]), aspect="auto", origin="lower", cmap="Blues", vmin=0, vmax=1)
    for (layer, position), token in np.ndenumerate(r.tokens[..., 0]):
        ax.text(position, layer, repr(token)[1:-1][:10], ha="center", va="center", fontsize=5)
    ax.set_xticks(range(len(words)), words, rotation=45, ha="right")
    ax.set_title(title)
axes[0].set_ylabel("layer (0 is the embedding, the last is the output)")
plt.show()

## More

[docs/USAGE.md](https://github.com/jianguda/lexframe/blob/main/docs/USAGE.md) lists every command, [examples/walkthrough.py](https://github.com/jianguda/lexframe/blob/main/examples/walkthrough.py) goes through them, and [examples/baselines.py](https://github.com/jianguda/lexframe/blob/main/examples/baselines.py) also plots how close each method comes to the model's own readout, against the ceiling that no fixed lens can pass.